In [5]:
import json
import os
import sys

import numpy as np
import pandas as pd
from rich import print

In [6]:
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(f"Current working directory: {os.getcwd()}")
sys.path.append("tools")
from tools import binning

Current working directory: /Volumes/X31/01.Projects/atem

In [7]:
areas = ["NW", "NE", "SE", "SW", "tielines"]
output_prefix = "merged"
dx = 50.0
istart_channel = 3
iend_channel = 25

criteria_rerr = 0.04
criteria_uncertainty = 0.04
floors_c = 0.05


def rebuild_merged_binned_frames(areas, dx=50.0):
    all_values = []
    all_values_std = []
    all_soundings = []

    times = None
    n_turns = None
    dheader = None
    picker = None

    for area in areas:
        tmp = binning(dx=dx, area=area)
        if times is None:
            times = tmp["times"]
            n_turns = tmp["n_turns"]
            dheader = tmp["dheader"]
            picker = tmp["picker"]

        all_values.extend(tmp["values"])
        all_values_std.extend(tmp["values_std"])
        all_soundings.extend(tmp["soundings"])

    if not all_values:
        raise ValueError("No binned data were reconstructed from the requested areas.")

    df_data_binned = pd.DataFrame(
        data=np.vstack(all_values),
        columns=["Line", "distance"] + picker[1:],
    )
    df_data_std_binned = pd.DataFrame(
        data=np.vstack(all_values_std),
        columns=["bheight"] + dheader,
    )

    for col in df_data_binned.columns:
        if col == "Line":
            df_data_binned[col] = df_data_binned[col].astype(str)
        else:
            df_data_binned[col] = pd.to_numeric(df_data_binned[col], errors="coerce")

    for col in df_data_std_binned.columns:
        df_data_std_binned[col] = pd.to_numeric(
            df_data_std_binned[col], errors="coerce"
        )

    return {
        "times": np.asarray(times, dtype=float),
        "soundings": [int(val) for val in all_soundings],
        "n_turns": float(n_turns),
        "dheader": list(dheader),
        "picker": list(picker),
        "df_data_binned": df_data_binned,
        "df_data_std_binned": df_data_std_binned,
    }


In [8]:
merged_data = rebuild_merged_binned_frames(areas=areas, dx=dx)

In [9]:
with open("outputs/binned_data.json", "w") as f:
    json.dump(
        {
            "times": merged_data["times"].tolist(),
            "soundings": merged_data["soundings"],
            "n_turns": merged_data["n_turns"],
            "dheader": merged_data["dheader"],
            "picker": merged_data["picker"],
            "data": merged_data["df_data_binned"].to_dict(orient="list"),
            "std": merged_data["df_data_std_binned"].to_dict(orient="list")
        },
        f,
        indent=4,
    )